# 10 Write a workflow recipe

*AUTOMATE*

**Question:** Can I make this run happen again, exactly the same way?

A recipe is a small, versioned **data file**. It names the CEPT operations to run, the
inputs each one needs, the artifacts each one writes, and where the workflow goes next
after each outcome. It carries no code, no loops, no retries and no conditions. It never
edits a Case value, widens a tolerance, or promotes a claim.

- Declare the inputs you will supply, and mark the ones you can omit as optional.
- Name the operations in order, and list what each one writes.
- Say what happens on success and what happens on failure. Both are required.
- List the artifacts that must exist for the workflow to report a pass.

An empty reason is refused. That is the point: the reasoning travels with the workflow,
so a reader six months from now can still see why each step exists.


## ⚙ Runtime — Confirm prerequisites

Use Python 3.10+ with an installed `cept`, or a caller-owned wheel via `CEPT_WHEEL_URL`
plus exact `CEPT_WHEEL_SHA256`; the wheel must provide CEPT and OpenDSS. No released PyPI
version is assumed. Jupyter is needed only to execute this notebook.


## ⚙ Setup — Initialize the teaching environment

Run the setup cell once to load helpers and check runtime dependencies.


In [1]:
#@title Setup — run once, then read the results below
from hashlib import sha256
from urllib.request import urlopen

_bootstrap_url = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/main/public/notebooks/_lesson.py"
_bootstrap = urlopen(_bootstrap_url, timeout=60).read()
if sha256(_bootstrap).hexdigest() != "7707385fd5fe60a9bec2e0bdd1bda33bc7b06bab71a8304dff0a7de86d23a0c6":
    raise ValueError("Lesson helper hash mismatch")
exec(compile(_bootstrap, "cept-lesson", "exec"), globals())


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0


Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.


## ▶ Run — Write the smallest recipe that is still honest

Four stages: check the Case, execute it, verify that exact run directory, then write the
receipt. Inputs arrive as explicit `name=value` pairs. Nothing here is picked by mtime, by
"latest", or by a glob. This is the whole document; the next cell runs a shipped one.


In [2]:
#@title Write recipes/mini.yaml
import pathlib

RECIPE_TEXT = """schema: workflow-recipe-v2
id: mini
version: 0.1.0
title: Check one Case, run it, verify it, write the receipt

needs:
  case:
    type: file
  run_dir:
    type: string

steps:
  - check.case case
  - run.study case run_dir
  - verify.study run_dir
  - write.receipt

must_produce:
  - case.json
  - results.json
  - manifest.json
  - validation_report.json
  - report.html

limits:
  - >-
    A completed workflow is not engineering validation and not project
    validation. The claim ceiling is WORKFLOW_VALIDATED.
"""
pathlib.Path("recipes").mkdir(exist_ok=True)
pathlib.Path("recipes/mini.yaml").write_text(RECIPE_TEXT, encoding="utf-8")
print(f"wrote recipes/mini.yaml, {len(RECIPE_TEXT.splitlines())} lines")


wrote recipes/mini.yaml, 28 lines


In [3]:
#@title Run it end to end
!cept run --demo load-flow \
    --network ieee13 \
    --out runs/mini-input \
    --force \
    --format text

!cept run --recipe run-audit \
    --input run_dir=runs/mini-input \
    --input case=runs/mini-input/case.json \
    --input audit_root=runs \
    --out runs/audit-out \
    --format text


CEPT study result: FINISHED
----------------------------
Result             Finished the balanced load flow and saved the evidence
Saved run          runs\mini-input
Case fingerprint   748c8026c9d6 (matches the case you ran)

What this means
  The study completed and saved solver-backed evidence.
  It does NOT approve a real project or field installation.

Next
  cept verify runs\mini-input --format text


environment: installed
cept module: <wheel env>\Lib\site-packages\cept\__init__.py
{
  "passed": true,
  "runs": [
    {
      "run_dir": ".\\runs\\mini-input",
      "status": "pass",
      "manifest_status": null,
      "claim": "WORKFLOW_VALIDATED",
      "reasons": [],
      "case_fingerprint": "748c8026c9d6",
      "identity_map_present": false,
      "semantic_registry_version": null,
      "edition": "public"
    }
  ]
}
recipe:   run-audit 1.0.0
run id:   1c2b48cf6b2acf259b40aa4ddac8fb82
status:   completed
passed:   true
  stage 1-verify.study: completed (verify.study)
  stage 2-audit.physics: completed (audit.physics)
  stage 3-audit.perunit: completed (audit.perunit)
  stage 4-audit.artifacts: completed (audit.artifacts)
  stage 5-write.receipt: completed (write.receipt)
claim:    workflow-verified: 5/5 stages completed, 4/4 expected artifacts hashed, literal passed=true in validation_report.json from stage '1-verify.study'
result:   .\runs\audit-out\recipe-run.json


## ▶ Run — Run a recipe that ships with the wheel

`run-audit` is one of the recipes in the wheel. Give it the run directory, its Case, and
the root to inventory. It verifies the run, audits it, and writes its own receipt.


In [4]:
#@title The same recipe, with one input that does not exist
import subprocess
import sys

_refused = subprocess.run(
    [
        sys.executable, "-m", "cept.recipes", "recipes/mini.yaml",
        "--input", "run_dir=runs/mini-input",
        "--input", "case=runs/mini-input/does-not-exist.json",
        "--input", "audit_root=runs",
        "--out", "runs/mini-refused-out",
    ],
    capture_output=True,
    text=True,
)
print("exit code:", _refused.returncode)
for _line in (_refused.stdout + _refused.stderr).strip().splitlines():
    if _line.strip():
        print(_line)


exit code: 1
environment: installed
cept module: <wheel env>\Lib\site-packages\cept\__init__.py
error: recipe input 'case' points at a missing file: runs/mini-input/does-not-exist.json (resolved to .\runs\mini-input\does-not-exist.json)
result:  .\runs\mini-refused-out\recipe-run.json


## ▶ Run — A recipe fails closed on a bad input

Point the same recipe at a file that is not there. It refuses and names the input, instead
of running the rest of the workflow against a default.

The receipt records what actually happened: every stage, its status, and the hashes of the
artifacts that were found. It is not a validation verdict, and it is not part of the run's
own hash-bound evidence set.


In [5]:
#@title Read the completion receipt
import json
import pathlib

_receipt = json.loads(pathlib.Path("runs/audit-out/recipe-run.json").read_text(encoding="utf-8"))
print("recipe: ", _receipt["recipe_id"], _receipt["recipe_version"])
print("run id: ", _receipt["run_id"])
print("status: ", _receipt["status"])
print("passed: ", _receipt["passed"])
print("claim:  ", _receipt["claim"])
print()
for _stage in _receipt["stages"]:
    print(f"  {_stage['stage_id']:<13} {_stage['status']:<10} {_stage['operation']}")


recipe:  run-audit 1.0.0
run id:  1c2b48cf6b2acf259b40aa4ddac8fb82
status:  completed
passed:  True
claim:   workflow-verified: 5/5 stages completed, 4/4 expected artifacts hashed, literal passed=true in validation_report.json from stage '1-verify.study'

  1-verify.study completed  verify.study
  2-audit.physics completed  audit.physics
  3-audit.perunit completed  audit.perunit
  4-audit.artifacts completed  audit.artifacts
  5-write.receipt completed  write.receipt


## ▶ Run — Read the receipt the workflow wrote

The receipt records what actually happened: every stage, its status, and the hashes of the
artifacts that were found. It is not a validation verdict, and it is not part of the run's
own hash-bound evidence set.

- `passed: true` means every declared stage completed and every declared artifact was
  found by exact name and hashed. It says the workflow ran as written.
- It does not say the physics is right, that another engine agrees, or that a project is
  validated. The ceiling is `WORKFLOW_VALIDATED`.
- A refused run leaves no receipt claiming a pass, so "the workflow finished" and "the
  workflow passed" stay different statements.
- Because the receipt names paths and digests rather than timestamps, two runs of the same
  recipe over the same inputs are comparable without asking which one is newer.


## ⓘ Interpret — What the receipt does and does not say

Add a retry to the document above and the file is refused before anything runs. The recipe
vocabulary has no loops, retries, parallelism or schedules, and no inline code: a recipe
declares *which* operation runs, and the operation itself owns *how*.


In [6]:
#@title A retry in a recipe is refused at parse time
import pathlib
import subprocess
import sys

_illegal = RECIPE_TEXT.replace(
    "    operation: run.study\n    owner: python\n",
    "    operation: run.study\n    owner: python\n    retry: 3\n",
)
pathlib.Path("recipes/illegal.yaml").write_text(_illegal, encoding="utf-8")

_refused = subprocess.run(
    [
        sys.executable, "-m", "cept.recipes", "recipes/illegal.yaml",
        "--input", "case=runs/mini-input/case.json",
        "--input", "run_dir=runs/mini-illegal",
        "--dry-run",
    ],
    capture_output=True,
    text=True,
)
print("exit code:", _refused.returncode)
for _line in (_refused.stdout + _refused.stderr).strip().splitlines():
    if _line.strip():
        print(_line)


exit code: 3
environment: installed
cept module: <wheel env>\Lib\site-packages\cept\__init__.py
recipe:   mini 0.1.0
run id:   0a71a286983fd9ee6efd3bbfa8aa1861
status:   blocked
passed:   false
  stage 1-check.case: skipped (check.case)
  stage 2-run.study: skipped (run.study)
  stage 3-verify.study: skipped (verify.study)
  stage 4-write.receipt: skipped (write.receipt)
claim:    plan-only: 4 stage(s) rendered, 0 executed
result:   .\recipe-run.json
